# Featured Event — Candidate Parsing & Drill-Down Debug Notebook

Loads the cached Brave candidates and walks through each pipeline stage so you can see exactly what's being kept, drilled, or dropped — and why.

**Prereq:** run the pipeline at least once with `BRAVE_CACHE_DIR` defaulting on (or set explicitly) so `Featured Event/Code/brave_cache/*.json` exists. This notebook reads from that cache.

In [8]:
import os, sys, json
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd

# Point Python at the project's shared code modules
REPO = Path.cwd().resolve()
while REPO.name and not (REPO / 'NewsletterCreation').exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'NewsletterCreation' / 'Code'))

from aggregator_drilldown import is_aggregator_url, drill_down_candidate, find_primary_url, _hostname
from event_date_filter import filter_candidates_by_date, upcoming_friday

CACHE_DIR = REPO / 'Featured Event' / 'Code' / 'brave_cache'
print('Cache dir:', CACHE_DIR)
print('Cache files:', [p.name for p in CACHE_DIR.glob('*.json')])

Cache dir: /Users/couch2coders/Documents/GitHub/newsletters/Featured Event/Code/brave_cache
Cache files: []


## 1. Pick a cached newsletter round to inspect

In [11]:
NEWSLETTER = 'East_Cobb_Connect'
ROUND      = 1

cache_path = CACHE_DIR / f'{NEWSLETTER}_round{ROUND}.json'
candidates = json.loads(cache_path.read_text(encoding='utf-8'))
print(f'Loaded {len(candidates)} candidates from {cache_path.name}')

df = pd.DataFrame([{
    'title': c.get('title', '')[:80],
    'host':  _hostname(c.get('url', '')),
    'url':   c.get('url', ''),
} for c in candidates])
df

FileNotFoundError: [Errno 2] No such file or directory: '/Users/couch2coders/Documents/GitHub/newsletters/Featured Event/Code/brave_cache/East_Cobb_Connect_round1.json'

## 2. Stage 1 — Listicle detection

Listicle pages ("5 things to do", "weekend checklist") get dropped *before* drilling because reducing a multi-event roundup to one primary URL causes mismatches downstream.

In [14]:
LISTICLE_MARKERS = (
    'things to do', 'things-to-do', '5 things', '10 things',
    'weekend checklist', 'weekend events', 'weekend roundup',
    'weekend guide', 'your weekend', 'events this weekend',
    'events this week', 'events you absolutely need',
    'out and about', 'what to do this', "what's happening",
    'upcoming events', 'calendar of events', 'events calendar',
    'fun things to do', 'guide to events', 'things to do in',
)

def is_listicle(c):
    blob = f"{c.get('title','')} {c.get('url','')}".lower()
    matched = [m for m in LISTICLE_MARKERS if m in blob]
    return matched

rows = []
for c in candidates:
    is_agg = is_aggregator_url(c.get('url', ''))
    matched = is_listicle(c) if is_agg else []
    rows.append({
        'title':      c.get('title','')[:60],
        'host':       _hostname(c.get('url','')),
        'aggregator': is_agg,
        'listicle':   bool(matched),
        'markers':    ', '.join(matched) if matched else '',
    })
pd.DataFrame(rows)

NameError: name 'candidates' is not defined

## 3. Stage 2 — Drill down each surviving aggregator URL

For every aggregator URL that **isn't** a listicle, fetch the page, score every outbound anchor by (URL-slug overlap × 4) + (heading overlap × 4) + (anchor-text overlap × 3), and swap the candidate's URL to the highest-scoring primary if it clears the threshold (MIN_SCORE = 4).

In [ ]:
drill_rows = []
for c in candidates:
    url = c.get('url', '')
    if not is_aggregator_url(url):
        continue
    if is_listicle(c):
        drill_rows.append({
            'title': c.get('title','')[:50],
            'original_host': _hostname(url),
            'verdict': 'LISTICLE — dropped pre-drill',
            'new_url': '',
        })
        continue
    # Make a copy so we don't mutate the cached candidate
    c2 = dict(c)
    drill_down_candidate(c2)
    drilled = c2.get('drilled', False)
    drill_rows.append({
        'title':         c.get('title','')[:50],
        'original_host': _hostname(url),
        'verdict':       'DRILLED' if drilled else 'KEPT ORIGINAL',
        'new_url':       c2.get('url', '') if drilled else url,
        'new_host':      _hostname(c2.get('url','') if drilled else url),
    })
pd.DataFrame(drill_rows)

## 4. Inspect drill scoring on a single URL

Edit `inspect_url` below to dig into why a particular drill picked the URL it did. This calls `find_primary_url` directly — same code path as the pipeline.

In [ ]:
inspect_url   = 'https://eastcobber.com/20th-annual-taste-of-east-cobb-2026/'
inspect_title = '20th Annual Taste of East Cobb 2026'

primary = find_primary_url(inspect_url, title=inspect_title)
print('Drill result:', primary or '(no primary chosen)')

## 5. Stage 3 — Date filter

After listicle drop + drill, the date filter checks title + summary + article body + primary-text for a date on or after the upcoming-Friday floor. Anything purely about past dates gets removed.

In [ ]:
# Build the post-listicle, post-drill pool the same way the pipeline does
pool = []
for c in candidates:
    url = c.get('url', '')
    if is_aggregator_url(url):
        if is_listicle(c):
            continue
        c2 = dict(c)
        drill_down_candidate(c2)
        pool.append(c2)
    else:
        pool.append(c)

floor = upcoming_friday()
print('Date floor:', floor)
kept, past_urls = filter_candidates_by_date(
    pool, floor,
    text_keys=('title', 'summary', 'article_text', 'primary_text'),
)

kept_urls = {c['url'] for c in kept}
rows = []
for c in pool:
    rows.append({
        'title':   c.get('title','')[:60],
        'host':    _hostname(c.get('url','')),
        'kept':    c.get('url') in kept_urls,
        'dates':   ', '.join(c.get('dates_found', [])) if c.get('dates_found') else '',
    })
pd.DataFrame(rows).sort_values('kept', ascending=False)

## 6. Final pool (what Claude sees)

In [ ]:
print(f'Final pool: {len(kept)} candidates\n')
for i, c in enumerate(kept, 1):
    print(f'{i}. {c.get("title","?")[:80]}')
    print(f'   {c.get("url","")}')
    print()